# Phase 13: Methodology-Clean JEPA Continuation Test

This local CUDA notebook answers the Phase 12 baseline concern directly.

Primary question: does JEPA continuation improve the released dSVA checkpoint compared with doing nothing, on the exact same broader victim panel?

Protocol:

- Evaluate the untouched released dSVA checkpoint on five victims: `resnet50`, `convnext_tiny`, `vit_b_16`, `efficientnet_b0`, `swin_t`.
- Run a JEPA-heavy continuation ablation from the released checkpoint: `jepa_weight` 0.2, 0.3, 0.5, 1.0 for 2 epochs.
- Keep matched DINO+MAE continuation controls as diagnostics, not as the main baseline.
- Train on 1000 Imagenette train images, evaluate on full Imagenette validation coverage via `eval_limit=5000`.
- Decide using gains vs untouched released dSVA first; matched continuation gains are secondary.


## Setup


In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'A local CUDA GPU is required for this experiment.' 


CUDA available: True
GPU: NVIDIA GeForce RTX 2060 SUPER


In [2]:
from pathlib import Path

cwd = Path.cwd()
candidates = [cwd, cwd.parent]
REPO_ROOT = next(
    (path for path in candidates if (path / 'scripts' / 'sweep_dsva_jepa_finetune.py').exists()),
    None,
)
assert REPO_ROOT is not None, f'Could not find repo root from {cwd}'
print('Repo root:', REPO_ROOT)


Repo root: d:\Florent\Desktop\jepa-transfer-attacks


In [3]:
TRAIN_ROOT = REPO_ROOT / 'imagenette2-320' / 'train'
VAL_ROOT = REPO_ROOT / 'imagenette2-320' / 'val'
DSVA_CHECKPOINT = REPO_ROOT / 'external' / 'models' / 'dSVA' / 'model.pth'

assert TRAIN_ROOT.exists(), f'Missing training data: {TRAIN_ROOT}'
assert VAL_ROOT.exists(), f'Missing validation data: {VAL_ROOT}'
assert DSVA_CHECKPOINT.exists(), f'Missing dSVA checkpoint: {DSVA_CHECKPOINT}'

print('Train root:', TRAIN_ROOT)
print('Val root:', VAL_ROOT)
print('dSVA checkpoint:', DSVA_CHECKPOINT)


Train root: d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\train
Val root: d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\val
dSVA checkpoint: d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth


## Configuration


In [4]:
SEEDS = [0, 1, 2]
TRAIN_LIMIT = 1000
EVAL_LIMIT = 5000
EPSILON = '0.06274509803921569'
LR = '0.00005'
EPOCHS = 2
OUTPUT_MODE = 'scaled-delta'
VICTIMS = ['resnet50', 'convnext_tiny', 'vit_b_16', 'efficientnet_b0', 'swin_t']
JEPA_CONFIGS = ['0.2:0.00005', '0.3:0.00005', '0.5:0.00005', '1.0:0.00005']

PHASE13_OUTPUT_DIR = REPO_ROOT / 'results' / 'phase13_jepa_weight_ablation'
PHASE13_ANALYSIS_DIR = REPO_ROOT / 'results' / 'phase13_analysis'
PHASE13_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE13_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PERF_FLAGS = ['--compile', '--compile-mode', 'reduce-overhead']
EVAL_PERF_FLAGS = ['--eval-amp', *TRAIN_PERF_FLAGS]

print('Output dir:', PHASE13_OUTPUT_DIR)
print('Analysis dir:', PHASE13_ANALYSIS_DIR)
print('Victims:', VICTIMS)
print('JEPA configs:', JEPA_CONFIGS)
print('Performance flags:', EVAL_PERF_FLAGS)


Output dir: d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_jepa_weight_ablation
Analysis dir: d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_analysis
Victims: ['resnet50', 'convnext_tiny', 'vit_b_16', 'efficientnet_b0', 'swin_t']
JEPA configs: ['0.2:0.00005', '0.3:0.00005', '0.5:0.00005', '1.0:0.00005']
Performance flags: ['--eval-amp', '--compile', '--compile-mode', 'reduce-overhead']


## Helpers


In [5]:
import subprocess, sys
from pathlib import Path

def run_checked(cmd):
    print('\n' + ' '.join(map(str, cmd)), flush=True)
    log_dir = REPO_ROOT / 'results' / 'phase13_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (Path(str(cmd[1])).stem + '_' + str(len(list(log_dir.glob('*.log')))) + '.log')
    with log_path.open('w', encoding='utf-8', errors='replace') as handle:
        result = subprocess.run(
            cmd,
            cwd=REPO_ROOT,
            text=True,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )
    text = log_path.read_text(encoding='utf-8', errors='replace')
    print(text[-12000:])
    print('Log:', log_path)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


## Smoke Test


In [6]:
smoke_csv = REPO_ROOT / 'results' / 'phase13_smoke_released_dsva_broader.csv'
if smoke_csv.exists():
    print('Skipping existing smoke CSV:', smoke_csv)
else:
    run_checked([
        sys.executable,
        'scripts/run_dsva_checkpoint_attack.py',
        '--data-root', str(VAL_ROOT),
        '--checkpoint', str(DSVA_CHECKPOINT),
        '--output-mode', 'adv',
        '--limit', '16',
        '--batch-size', '8',
        '--epsilon', EPSILON,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--output-csv', str(smoke_csv),
        '--amp',
    ])



c:\Users\Florent\anaconda3\python.exe scripts/run_dsva_checkpoint_attack.py --data-root d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\val --checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-mode adv --limit 16 --batch-size 8 --epsilon 0.06274509803921569 --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --output-csv d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_smoke_released_dsva_broader.csv --amp

dSVA checkpoint transfer: 100%|██████████| 2/2 [00:11<00:00,  5.68s/it]
| model | n | clean_acc | adv_acc | acc_drop | attack_success_rate |
| --- | ---: | ---: | ---: | ---: | ---: |
| resnet50 | 16 | 93.75% | 37.50% | 56.25% | 60.00% |
| convnext_tiny | 16 | 93.75% | 43.75% | 50.00% | 53.33% |
| vit_b_16 | 16 | 100.00% | 6.25% | 93.75% | 93.75% |
| efficientnet_b0 | 16 | 93.75% | 12.50% | 81.25% | 86.67% |
| swin_t | 16 | 93.75% | 50.00% | 43.75% | 46.67% |

Mean transfer success: 68.08%
Max L_in

## Untouched Released dSVA Broader Baseline


In [7]:
untouched_csv = PHASE13_ANALYSIS_DIR / 'official_dsva_broader_eval.csv'
if untouched_csv.exists():
    print('Skipping existing untouched baseline:', untouched_csv)
else:
    run_checked([
        sys.executable,
        'scripts/run_dsva_checkpoint_attack.py',
        '--data-root', str(VAL_ROOT),
        '--checkpoint', str(DSVA_CHECKPOINT),
        '--output-mode', 'adv',
        '--limit', str(EVAL_LIMIT),
        '--batch-size', '8',
        '--epsilon', EPSILON,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--output-csv', str(untouched_csv),
        '--amp',
    ])



c:\Users\Florent\anaconda3\python.exe scripts/run_dsva_checkpoint_attack.py --data-root d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\val --checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-mode adv --limit 5000 --batch-size 8 --epsilon 0.06274509803921569 --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --output-csv d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_analysis\official_dsva_broader_eval.csv --amp
91 [01:47<00:38,  3.90it/s]
dSVA checkpoint transfer: 100%|██████████| 491/491 [02:38<00:00,  3.10it/s]
| model | n | clean_acc | adv_acc | acc_drop | attack_success_rate |
| --- | ---: | ---: | ---: | ---: | ---: |
| resnet50 | 3925 | 87.34% | 27.92% | 59.41% | 68.90% |
| convnext_tiny | 3925 | 87.59% | 40.03% | 47.57% | 55.44% |
| vit_b_16 | 3925 | 91.85% | 21.12% | 70.73% | 77.36% |
| efficientnet_b0 | 3925 | 86.04% | 4.82% | 81.22% | 94.61% |
| swin_t | 3925 | 84.66% | 42.88% | 41.78% 

In [8]:
import pandas as pd

untouched = pd.read_csv(untouched_csv)
untouched_mean = untouched['attack_success_rate'].mean()
print('Untouched broader mean:', f'{100 * untouched_mean:.2f}%')
untouched


Untouched broader mean: 69.44%


,model,n,clean_acc,adv_acc,acc_drop,attack_success_rate
0,resnet50,3925,0.873376,0.279236,0.594140,0.689032
1,convnext_tiny,3925,0.875924,0.400255,0.475669,0.554392
2,vit_b_16,3925,0.918471,0.211210,0.707261,0.773648
3,efficientnet_b0,3925,0.860382,0.048153,0.812229,0.946106
4,swin_t,3925,0.846624,0.428790,0.417834,0.508878


## JEPA-Heavy Weight Ablation


In [9]:
for seed in SEEDS:
    seed_dir = PHASE13_OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', str(TRAIN_ROOT),
        '--val-root', str(VAL_ROOT),
        '--init-checkpoint', str(DSVA_CHECKPOINT),
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase13_seed{seed}_jepa_heavy_ep{EPOCHS}',
        '--configs', *JEPA_CONFIGS,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--epochs', str(EPOCHS),
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epsilon', EPSILON,
        '--output-mode', OUTPUT_MODE,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
        *EVAL_PERF_FLAGS,
    ]
    run_checked(cmd)



c:\Users\Florent\anaconda3\python.exe scripts/sweep_dsva_jepa_finetune.py --train-root d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\train --val-root d:\Florent\Desktop\jepa-transfer-attacks\imagenette2-320\val --init-checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-dir d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_jepa_weight_ablation\seed_0 --run-prefix phase13_seed0_jepa_heavy_ep2 --configs 0.2:0.00005 0.3:0.00005 0.5:0.00005 1.0:0.00005 --limit 1000 --eval-limit 5000 --epochs 2 --batch-size 1 --eval-batch-size 8 --grad-accum-steps 8 --epsilon 0.06274509803921569 --output-mode scaled-delta --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --seed 0 --normalize-loss-weights --skip-existing --eval-amp --compile --compile-mode reduce-overhead
dSVA checkpoint transfer: 100%|██████████| 491/491 [02:44<00:00,  2.98it/s]
| model | n | clean_acc | adv_acc | acc_drop | attack_success_rate |
| --- | 

## Aggregate Results


In [10]:
run_checked([
    sys.executable,
    'scripts/analyze_phase11_scale_results.py',
    '--scale-root', 'results/phase13_jepa_weight_ablation',
    '--output-detail-csv', 'results/phase13_analysis/jepa_weight_detail.csv',
    '--output-aggregate-csv', 'results/phase13_analysis/jepa_weight_aggregate.csv',
])

aggregate = pd.read_csv(PHASE13_ANALYSIS_DIR / 'jepa_weight_aggregate.csv')
aggregate



c:\Users\Florent\anaconda3\python.exe scripts/analyze_phase11_scale_results.py --scale-root results/phase13_jepa_weight_ablation --output-detail-csv results/phase13_analysis/jepa_weight_detail.csv --output-aggregate-csv results/phase13_analysis/jepa_weight_aggregate.csv
Wrote detail CSV: results\phase13_analysis\jepa_weight_detail.csv
Wrote aggregate CSV: results\phase13_analysis\jepa_weight_aggregate.csv

Log: d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_logs\analyze_phase11_scale_results_5.log


,objectives,run_type,jepa_weight,lr,epochs,train_limit,num_seeds,mean_transfer_success,std_transfer_success,resnet50_mean,resnet50_std,convnext_tiny_mean,convnext_tiny_std,vit_b_16_mean,vit_b_16_std,efficientnet_b0_mean,efficientnet_b0_std,swin_t_mean,swin_t_std
0,dino_mae,control,0.0,0.00005,2,1000,3,0.680356,0.010273,0.707118,0.020149,0.522106,0.015731,0.738141,0.002883,0.947608,0.008334,0.486809,0.010435
1,dino_mae_jepa,jepa,0.2,0.00005,2,1000,3,0.702227,0.008969,0.718300,0.022449,0.543242,0.008810,0.775589,0.002466,0.949176,0.009077,0.524827,0.007824
2,dino_mae_jepa,jepa,0.3,0.00005,2,1000,3,0.706548,0.007763,0.715869,0.023303,0.548866,0.007053,0.783819,0.001577,0.951535,0.008791,0.532651,0.001379
3,dino_mae_jepa,jepa,0.5,0.00005,2,1000,3,0.706443,0.009216,0.714119,0.024276,0.550999,0.006408,0.786408,0.005541,0.945232,0.011553,0.535460,0.008363
4,dino_mae_jepa,jepa,1.0,0.00005,2,1000,3,0.698749,0.014093,0.702742,0.023755,0.532480,0.015431,0.788534,0.011125,0.942853,0.010964,0.527134,0.010028


## Compare Against Untouched and Matched Controls


In [11]:
rows = aggregate.copy()
rows['mean_transfer_success'] = rows['mean_transfer_success'].astype(float)
controls = rows[rows['run_type'] == 'control'].set_index(['lr', 'epochs', 'train_limit'])['mean_transfer_success']

jepa = rows[rows['run_type'] == 'jepa'].copy()
jepa['control_mean'] = jepa.apply(
    lambda row: controls.loc[(row['lr'], row['epochs'], row['train_limit'])],
    axis=1,
)
jepa['gain_vs_control'] = jepa['mean_transfer_success'] - jepa['control_mean']
jepa['gain_vs_untouched'] = jepa['mean_transfer_success'] - untouched_mean

victim_baseline = untouched.set_index('model')['attack_success_rate'].to_dict()
for victim in VICTIMS:
    column = f'{victim}_mean'
    if column in jepa.columns and victim in victim_baseline:
        jepa[f'{victim}_gain_vs_untouched'] = jepa[column].astype(float) - victim_baseline[victim]

ordered = jepa.sort_values(['gain_vs_untouched', 'mean_transfer_success'], ascending=False)
ordered


,objectives,run_type,jepa_weight,lr,epochs,train_limit,num_seeds,mean_transfer_success,std_transfer_success,resnet50_mean,...,swin_t_mean,swin_t_std,control_mean,gain_vs_control,gain_vs_untouched,resnet50_gain_vs_untouched,convnext_tiny_gain_vs_untouched,vit_b_16_gain_vs_untouched,efficientnet_b0_gain_vs_untouched,swin_t_gain_vs_untouched
2,dino_mae_jepa,jepa,0.3,0.00005,2,1000,3,0.706548,0.007763,0.715869,...,0.532651,0.001379,0.680356,0.026192,0.012137,0.026837,-0.005526,0.010171,0.005429,0.023773
3,dino_mae_jepa,jepa,0.5,0.00005,2,1000,3,0.706443,0.009216,0.714119,...,0.535460,0.008363,0.680356,0.026087,0.012032,0.025087,-0.003393,0.012760,-0.000874,0.026582
1,dino_mae_jepa,jepa,0.2,0.00005,2,1000,3,0.702227,0.008969,0.718300,...,0.524827,0.007824,0.680356,0.021871,0.007816,0.029268,-0.011150,0.001941,0.003070,0.015949
4,dino_mae_jepa,jepa,1.0,0.00005,2,1000,3,0.698749,0.014093,0.702742,...,0.527134,0.010028,0.680356,0.018393,0.004338,0.013710,-0.021912,0.014886,-0.003253,0.018256


## Decision Read


In [12]:
best = ordered.iloc[0]
print('Best JEPA weight:', best['jepa_weight'])
print('Mean transfer:', f"{100 * best['mean_transfer_success']:.2f}%")
print('Gain vs untouched:', f"{100 * best['gain_vs_untouched']:+.2f} pts")
print('Gain vs matched control:', f"{100 * best['gain_vs_control']:+.2f} pts")

if best['gain_vs_untouched'] >= 0.01:
    print('Decision: strong enough to consider larger-scale validation.')
elif best['gain_vs_untouched'] > 0:
    print('Decision: positive but modest; fix continuation fidelity or add per-image complementarity before scaling.')
else:
    print('Decision: do not scale; JEPA did not beat the untouched released checkpoint on this panel.')


Best JEPA weight: 0.3
Mean transfer: 70.65%
Gain vs untouched: +1.21 pts
Gain vs matched control: +2.62 pts
Decision: strong enough to consider larger-scale validation.


## Package CSV Outputs


In [13]:
import tarfile

archive_path = REPO_ROOT / 'results' / 'phase13_csv_artifacts.tar.gz'
paths = list(PHASE13_OUTPUT_DIR.rglob('*.csv'))
paths += list(PHASE13_ANALYSIS_DIR.rglob('*.csv'))
smoke_csv = REPO_ROOT / 'results' / 'phase13_smoke_released_dsva_broader.csv'
if smoke_csv.exists():
    paths.append(smoke_csv)

with tarfile.open(archive_path, 'w:gz') as tar:
    for path in paths:
        tar.add(path, arcname=path.relative_to(REPO_ROOT))
print('Archive:', archive_path)


Archive: d:\Florent\Desktop\jepa-transfer-attacks\results\phase13_csv_artifacts.tar.gz
